# CRM Acess Governance & Customer Data PRotection
## Exploratory Data Analysis (EDA) - V.2

Este notebook é a etapa exploratória do projeto, onde podemos entender a base de dados

A base de dados é pública e pode ser adquirida no link: https://www.kaggle.com/datasets/dataset066/permission-aware-crm-governance-synthetic-dataset

Representa eventis de acesso a um ambiente de CRM e contém informações sobre perfil do usuário, ação realizada, dispositivo, horario, senbilidade do dado, permissões, comportamento de login, scores de risco/governança e decisão final de acesso.

## Objetivos do EAD

1. Validar estrutura e qualidade dos dados.
2. Entender a distribuição das variáveis numéricas e categóricas.
3. Analisar comportamento de acesso e fatores de risco.
4. Investigar relações entre `Permission_Granted` e `Access_Decision`.
5. Identificar fatores associados a bloqueios e revisões.
6. Explorar interações entre perfil, ação, dispositivo, sensibilidade e comportamento.
7. Detectar valores extremos e padrões incomuns.
8. Gerar hipóteses para a futura política de governança.
9. Preparar métricas e dimensões úteis para o modelo analítico e Power BI.

In [1]:
# Importação de bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("Permission_Aware_CRM_Governance_Synthetic_50000.csv")

## 2. Carregamento e Data Discovery

In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]:,}")
print(df.head())

Linhas: 50,000
Colunas: 15
   User_ID       Role Region Lead_Source CRM_Action  Daily_Logins  \
0        1      Admin  North    Referral   ViewLead             6   
1        2  Sales Rep   West         Web   ViewLead             5   
2        3  Sales Rep  South     Partner   EditLead             6   
3        4    Analyst  South    Campaign   ViewLead            10   
4        5  Sales Rep   West         Web   EditLead             6   

   Failed_Logins  Access_Hour Device_Type  Data_Sensitivity  \
0              1           10     Managed                 5   
1              1           12     Managed                 5   
2              1           20     Managed                 1   
3              0           10     Managed                 3   
4              2           17     Managed                 1   

   Policy_Compliance_Score  Anomaly_Score  Permission_Granted  \
0                    76.19           0.11                True   
1                    62.87           0.46        

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   User_ID                  50000 non-null  int64  
 1   Role                     50000 non-null  str    
 2   Region                   50000 non-null  str    
 3   Lead_Source              50000 non-null  str    
 4   CRM_Action               50000 non-null  str    
 5   Daily_Logins             50000 non-null  int64  
 6   Failed_Logins            50000 non-null  int64  
 7   Access_Hour              50000 non-null  int64  
 8   Device_Type              50000 non-null  str    
 9   Data_Sensitivity         50000 non-null  int64  
 10  Policy_Compliance_Score  50000 non-null  float64
 11  Anomaly_Score            50000 non-null  float64
 12  Permission_Granted       50000 non-null  bool   
 13  Governance_Score         50000 non-null  float64
 14  Access_Decision          50000 no

In [5]:
structural_summary = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "Non_Null": df.notna().sum(),
    "nulls": df.isna().sum(),
    "Null_%": (df.isna().mean() * 100).round(2),
    "Unique_values": df.nunique(),
    "Cardinality_%": (df.nunique() / len(df) * 100).round(2) 
})

display(structural_summary)

,data_type,Non_Null,nulls,Null_%,Unique_values,Cardinality_%
User_ID,int64,50000,0,0.00,50000,100.00
Role,str,50000,0,0.00,5,0.01
Region,str,50000,0,0.00,4,0.01
Lead_Source,str,50000,0,0.00,5,0.01
CRM_Action,str,50000,0,0.00,6,0.01
Daily_Logins,int64,50000,0,0.00,21,0.04
Failed_Logins,int64,50000,0,0.00,7,0.01
Access_Hour,int64,50000,0,0.00,17,0.03
Device_Type,str,50000,0,0.00,2,0.00
Data_Sensitivity,int64,50000,0,0.00,5,0.01


In [6]:


low_cardinality_cols = [
    col for col in df.columns
    if df[col].nunique() <= 20
]

for col in low_cardinality_cols:
    print(f"\n{col} ({df[col].nunique()} valores únicos)")
    print(df[col].unique())



Role (5 valores únicos)
<StringArray>
['Admin', 'Sales Rep', 'Analyst', 'Support', 'Manager']
Length: 5, dtype: str

Region (4 valores únicos)
<StringArray>
['North', 'West', 'South', 'East']
Length: 4, dtype: str

Lead_Source (5 valores únicos)
<StringArray>
['Referral', 'Web', 'Partner', 'Campaign', 'Email']
Length: 5, dtype: str

CRM_Action (6 valores únicos)
<StringArray>
[         'ViewLead',          'EditLead', 'CreateOpportunity',
         'ExportCRM',        'DeleteLead',   'ApproveDiscount']
Length: 6, dtype: str

Failed_Logins (7 valores únicos)
[1 0 2 4 3 5 6]

Access_Hour (17 valores únicos)
[10 12 20 17 15  8 18 14  7 16 13 22 11 19  9 21  6]

Device_Type (2 valores únicos)
<StringArray>
['Managed', 'BYOD']
Length: 2, dtype: str

Data_Sensitivity (5 valores únicos)
[5 1 3 2 4]

Permission_Granted (2 valores únicos)
[ True False]

Access_Decision (2 valores únicos)
<StringArray>
['Block', 'Review']
Length: 2, dtype: str


## 2.5 Dicionário inicial dos dados

| Campo | Interpretação | Categoria analítica |
|---|---|---|
| `User_ID` | Identificador do usuário | Identificador |
| `Role` | Perfil/função do usuário | Controle de acesso |
| `Region` | Região associada ao registro | Dimensão |
| `Lead_Source` | Origem comercial do lead | Dimensão |
| `CRM_Action` | Ação realizada/solicitada | Atividade |
| `Daily_Logins` | Quantidade de logins diários | Comportamento |
| `Failed_Logins` | Tentativas falhas | Segurança |
| `Access_Hour` | Hora do acesso | Contexto temporal |
| `Device_Type` | Tipo de dispositivo | Segurança |
| `Data_Sensitivity` | Nível de sensibilidade | Governança |
| `Policy_Compliance_Score` | Aderência às políticas | Governança |
| `Anomaly_Score` | Grau de comportamento anômalo | Segurança |
| `Permission_Granted` | Existência de permissão funcional | Autorização |
| `Governance_Score` | Score agregado de governança | Governança |
| `Access_Decision` | Resultado do acesso | Outcome |

Este dicionário será refinado mais adiante para uma **Data Classification Matrix**.